# Neuronal Tuning Summary

This notebook has two complementary entry points over the same pre-computed `triage_stats` block:

1. **Cross-session / cross-condition aggregator** (run first). Collapses same-day duplicate units across sessions and conditions into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`, preserving per-session evidence so consistency of tuning can be inspected across repeats. Reads session-list `.txt` files (one path per line) and the authoritative `unit_catalog.csv` for anatomy.
2. **Per-session triage** (run after, or on its own). For each session root it reads every `*_tuning_curves_data.pkl` under `<session_root>/ephys/tuning_curves/`, applies thresholds to each cluster's `triage_stats`, and writes a timestamped per-session JSON listing flagged clusters by modality / direction / role.

The triage stats themselves are produced by `generate-rm` — this notebook is a pure pkl-to-{pickle,JSON} pass and never touches spike or USV data. Thresholds default to `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block) and can be overridden in the configuration cell below.


In [ ]:
### Imports

from __future__ import annotations

import json
from pathlib import Path

from usv_playpen.analyses.detect_interesting_tuning_neurons import (
    aggregate_units_across_conditions,
    detect_interesting_clusters,
)
from usv_playpen.os_utils import configure_path


## Configure session roots and thresholds

Each entry in `SESSION_ROOTS` produces ONE JSON file at `<session>/ephys/tuning_curves/interesting_neurons_<YYYYMMDD>_<HHMMSS>.json`. Sessions are processed sequentially.

Thresholds default to the values in `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block); override here if you want to sweep different values across the same set of pkls without re-computing tuning.

In [ ]:
SESSION_ROOTS = [
    # "/Volumes/falkner/Bartul/Data/20250919_145712",
    # "/Volumes/falkner/Bartul/Data/20250919_152924",
]

THRESHOLDS = {
    "z_threshold":               3.0,
    "min_consecutive_bins":        3,
    "vmi_alpha":                0.01,
    "vmi_min_bouts":              10,
    "spatial_info_bps_threshold": 0.5,
}


## Cross-session / cross-condition aggregator

Collapses same-day duplicate units across the sessions in `CONDITION_TO_SESSION_LIST` into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`. Each unit carries its identity, the catalog `anatomy_region`, and a `conditions` block — one entry per condition listing, per modality, `n_significant`, `n_tested`, `consistency`, an aggregate scalar, and per-session evidence rows.

Each value of `CONDITION_TO_SESSION_LIST` is a path to a `.txt` file with one session root per line (the `ephys_courtship_*_sessions_list.txt` lists under `/mnt/falkner/Bartul/modeling/input_files/`). Sessions missing a `tuning_curves` directory are recorded under `sessions_skipped` and not counted. Anatomy is read from the authoritative `unit_catalog.csv`; orphan pkls (no catalog row) raise.

The output is written to `<out_dir>/unit_triage_<YYYYMMDD>_<HHMMSS>.pkl`. Thresholds default to the same `detect_interesting_tuning_neurons` block in `analyses_settings.json` used by the per-session triage below, but can be overridden via keyword arguments to `aggregate_units_across_conditions(...)`.

In [ ]:
CONDITION_TO_SESSION_LIST = {
    "intact_female": "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_intact_partners_sessions_list.txt",
    "mute_female":   "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_mute_female_sessions_list.txt",
}

CATALOG_PATH = "/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"
AGGREGATOR_OUT_DIR = "/mnt/falkner/Bartul/neuronal_tuning"
DATA_ROOT = "/mnt/falkner/Bartul/Data"


In [ ]:
aggregator_pkl_path = aggregate_units_across_conditions(
    condition_to_session_list=CONDITION_TO_SESSION_LIST,
    catalog_path=CATALOG_PATH,
    out_dir=AGGREGATOR_OUT_DIR,
    data_root=DATA_ROOT,
    z_threshold=THRESHOLDS["z_threshold"],
    min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
    vmi_alpha=THRESHOLDS["vmi_alpha"],
    vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
    spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
    message_output=print,
)

print(f"\nAggregator wrote: {aggregator_pkl_path}")


## Run the triage, one session at a time

Each call to `detect_interesting_clusters` writes its own timestamped JSON next to the cluster pkls, so re-running this cell does not overwrite previous JSONs. The dict `results` collects the output paths per session for downstream inspection.

In [ ]:
results: dict[str, Path | None] = {}

for root in SESSION_ROOTS:
    resolved = configure_path(root)
    print(f"\n=== {resolved} ===")
    out_path = detect_interesting_clusters(
        root_directory=resolved,
        z_threshold=THRESHOLDS["z_threshold"],
        min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
        vmi_alpha=THRESHOLDS["vmi_alpha"],
        vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
        spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
        message_output=print,
    )
    results[resolved] = out_path

n_written = sum(p is not None for p in results.values())
print(f"\nDone. Wrote {n_written} / {len(SESSION_ROOTS)} JSON file(s).")


## Quick per-session peek (optional)

Loads one of the just-written JSONs and prints a one-screen summary: cluster counts, thresholds used, and the per-modality flag tally. Use this to sanity-check the run; the proper plots / aggregation will live in a later notebook section.

In [ ]:
session_to_inspect = SESSION_ROOTS[0] if SESSION_ROOTS else None
if session_to_inspect:
    json_path = results[configure_path(session_to_inspect)]
    if json_path is None:
        print(f"No JSON written for {session_to_inspect} (no pkls or no tuning_curves dir).")
    else:
        with open(json_path) as f:
            d = json.load(f)
        print(f"Session:           {d['session_root']}")
        print(f"Generated:         {d['generated_at']}")
        print(f"Thresholds:        {d['thresholds_used']}")
        print(f"Clusters total:    {d['n_clusters_total']}")
        print(f"Clusters flagged:  {d['n_clusters_flagged']}")
        print(f"Skipped no-triage: {d['n_clusters_skipped_no_triage']}")
        print()
        print(f"Modality flag tallies ({len(d['by_modality'])} modalities flagged):")
        for k in sorted(d['by_modality'].keys(), key=lambda kk: -len(d['by_modality'][kk]))[:25]:
            print(f"  {len(d['by_modality'][k]):4d}  {k}")
else:
    print("Add at least one session root to SESSION_ROOTS above.")
